In [2]:
pip install nlpaug sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 13.0 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix # Added import
import nlpaug.augmenter.word as naw
import random

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
dataset = load_dataset("ailsntua/QEvasion")

# Prepare labels
labels = dataset['train'].unique('clarity_label')
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

def add_labels(example):
    example['labels'] = label2id[example['clarity_label']]
    return example

dataset = dataset.map(add_labels)
dataset = dataset.remove_columns([
    col for col in dataset['train'].column_names if col not in ['question', 'interview_answer', 'labels']
])

print("Dataset ready:")
print(dataset)
print(f"Labels mapped: {label2id}")



# Contextual word replacement augmenter
aug = naw.ContextualWordEmbsAug(
    model_path='bert-base-uncased',
    action="substitute",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

def augment_text(text):
    if random.random() < 0.20:
        try:
            augmented = aug.augment(text)

            # nlpaug sometimes returns a list even for single input
            if isinstance(augmented, list):
                augmented = augmented[0]

            # Ensure we always return a string
            return str(augmented)

        except Exception:
            # In case augmentation fails
            return text

    return text


def augment_examples(example):
    """Apply augmentation ONLY to train split."""
    example['question'] = augment_text(example['question'])
    example['interview_answer'] = augment_text(example['interview_answer'])
    return example




Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

The following layers were not sharded: bert.encoder.layer.*.attention.self.key.weight, bert.encoder.layer.*.intermediate.dense.weight, bert.encoder.layer.*.attention.self.query.bias, cls.predictions.transform.dense.weight, bert.encoder.layer.*.intermediate.dense.bias, cls.predictions.decoder.weight, bert.embeddings.LayerNorm.weight, bert.encoder.layer.*.output.dense.bias, bert.embeddings.token_type_embeddings.weight, bert.encoder.layer.*.output.LayerNorm.weight, bert.encoder.layer.*.attention.output.dense.weight, bert.embeddings.position_embeddings.weight, cls.predictions.transform.LayerNorm.bias, cls.predictions.bias, bert.encoder.layer.*.attention.self.value.bias, bert.encoder.layer.*.output.LayerNorm.bias, bert.encoder.layer.*.output.dense.weight, bert.encoder.layer.*.attention.self.query.weight, bert.embeddings.word_embeddings.weight, cls.predictions.transform.LayerNorm.weight, bert.embeddings.LayerNorm.bias, bert.encoder.layer.*.attention.output.LayerNorm.bias, bert.encoder.layer.

In [6]:
# Apply augmentation to train set only
dataset["train"] = dataset["train"].map(augment_examples)

Map:   0%|          | 0/3448 [00:00<?, ? examples/s]

In [ ]:
dataset["train"].save_to_disk("QEvasion_train_augmented")

In [5]:
from datasets import load_from_disk

dataset = load_from_disk("QEvasion_augmented_dataset")
print(dataset)

FileNotFoundError: Directory QEvasion_augmented_dataset not found

In [7]:
# Calculate class weights for imbalanced data
def get_class_weights(dataset, num_labels):
    label_counts = Counter(dataset["train"]["labels"])
    total_samples = len(dataset["train"])

    # Calculate class weights (inverse frequency)
    class_weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)  # avoid division by zero
        weight = total_samples / (num_labels * count)
        class_weights.append(weight)

    # Move the tensor to the active device (GPU)
    return torch.tensor(
        class_weights, dtype=torch.float32, device=device
    )

    # Get class weights
class_weights = get_class_weights(dataset, num_labels)
print(f"Class weights: {class_weights}")
print(f"Class weights device: {class_weights.device}")  # Verify it's on CUDA



Class weights: tensor([1.0925, 0.5634, 3.2285], device='cuda:0')
Class weights device: cuda:0


In [8]:
# Focal Loss implementation :cite[1]:cite[8]
class FocalLoss(nn.Module):
    """
    Multi-class Focal loss implementation
    Focal loss helps address class imbalance by focusing on hard examples

    Args:
        gamma (float): Focusing parameter, higher values put more focus on hard examples
        weight (Tensor): Class weights tensor for handling imbalanced data
        ignore_index (int): Index to ignore in loss calculation
    """
    def __init__(self, gamma=1.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        # Calculate cross entropy loss
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)

        # Get probabilities
        pt = torch.exp(-ce_loss)

        # Compute focal loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()

In [9]:
# Updated Custom Trainer with corrected compute_loss signature
class CustomTrainer(Trainer):
    """
    Custom trainer that uses Focal Loss with class weights
    This subclass overrides the compute_loss method to use our custom loss function
    """

    def __init__(self, *args, class_weights=None, focal_gamma=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        # Extract labels and run model forward pass
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Compute focal loss with class weights
        loss = self.focal_loss(logits, labels)

        # Handle return_outputs as required by the Trainer
        return (loss, outputs) if return_outputs else loss

In [11]:
# Tokenization and model setup
model_checkpoint = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples['question'],
        examples['interview_answer'],
        truncation=True,
        padding="max_length",
        max_length=1680
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    # Use macro averaging for balanced metrics across classes
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Keep weighted for comparison
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted
    }

# --- MODIFICATION 1: Updated TrainingArguments ---
# We now evaluate, save, and load the best model based on 'f1_macro'
training_args = TrainingArguments(
    output_dir="ModernBERT_QEvasion_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",          # <--- MODIFIED (was "no")
    save_strategy="epoch",
    load_best_model_at_end=True,    # <--- MODIFIED (was False)
    metric_for_best_model="f1_macro", # <--- NEW
    greater_is_better=True,         # <--- NEW
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,  # Enable mixed precision (reduces memory usage)
    gradient_checkpointing=True,
)

# --- MODIFICATION 2: Updated CustomTrainer instantiation ---
# We pass the test set to eval_dataset
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],  # <--- NEW
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,  # Pass the calculated class weights
    focal_gamma=1.0,  # You can adjust this parameter
)

# trainer.train(resume_from_checkpoint=True)
print(f"Using class weights: {class_weights}")
print("Starting training with Focal Loss (evaluating on test set after each epoch)...")
trainer.train()

print("Training completed!")

# --- MODIFICATION 3: Updated Final Evaluation ---
# This will now evaluate the *best* model saved during training
# (because of load_best_model_at_end=True)
test_results = trainer.evaluate() # <--- MODIFIED (no arg needed)
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (from best epoch: {trainer.state.best_model_checkpoint})") # <--- MODIFIED
print("="*60)
for key, value in test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# Optional: Get detailed predictions
# This will also use the best model
print("\nDetailed predictions analysis (from best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                          target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


Map:   0%|          | 0/3448 [00:00<?, ? examples/s]

Map:   0%|          | 0/308 [00:00<?, ? examples/s]

The following layers were not sharded: model.final_norm.weight, head.dense.weight, model.layers.*.mlp_norm.weight, model.embeddings.tok_embeddings.weight, classifier.bias, model.layers.*.attn.Wo.weight, model.layers.*.mlp.Wi.weight, model.embeddings.norm.weight, head.norm.weight, model.layers.*.mlp.Wo.weight, classifier.weight, model.layers.*.attn.Wqkv.weight, model.layers.*.attn_norm.weight
Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1834643609.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and gener

Using class weights: tensor([1.0925, 0.5634, 3.2285], device='cuda:0')
Starting training with Focal Loss (evaluating on test set after each epoch)...


W1119 17:39:25.335000 1623 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
1,0.664300,0.558620,0.272727,0.275069,0.136188,0.210068,0.094977,0.467437,0.272727
2,0.634700,0.498622,0.558442,0.506593,0.572108,0.483468,0.604260,0.565083,0.558442
3,0.466200,0.634715,0.467532,0.447851,0.483607,0.462988,0.585631,0.502094,0.467532
4,0.325000,0.890214,0.568182,0.464558,0.573073,0.458691,0.578862,0.471879,0.568182
5,0.197700,0.861724,0.467532,0.448666,0.479425,0.464288,0.601091,0.525373,0.467532
6,0.176800,1.103414,0.535714,0.470457,0.554467,0.464835,0.610672,0.512794,0.535714
7,0.125300,1.395207,0.581169,0.466182,0.586681,0.478328,0.597494,0.463139,0.581169
8,0.058000,1.520726,0.597403,0.486120,0.602878,0.489931,0.611871,0.486706,0.597403


Training completed!



FINAL TEST RESULTS (from best epoch: ModernBERT_QEvasion_model/checkpoint-862)
eval_loss: 0.4986
eval_accuracy: 0.5584
eval_f1_macro: 0.5066
eval_f1_weighted: 0.5721
eval_precision_macro: 0.4835
eval_precision_weighted: 0.6043
eval_recall_macro: 0.5651
eval_recall_weighted: 0.5584

Detailed predictions analysis (from best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.36      0.46      0.40        79
     Ambivalent       0.72      0.59      0.65       206
Clear Non-Reply       0.37      0.65      0.47        23

       accuracy                           0.56       308
      macro avg       0.48      0.57      0.51       308
   weighted avg       0.60      0.56      0.57       308


Confusion Matrix:
[[ 36  38   5]
 [ 64 121  21]
 [  0   8  15]]


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the best model
drive_model_path = "/content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma"

print("=" * 60)
print("SAVING BEST MODEL TO GOOGLE DRIVE")
print("=" * 60)
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best F1 macro: {trainer.state.best_metric:.4f}")

# Save model and tokenizer
trainer.save_model(drive_model_path)
tokenizer.save_pretrained(drive_model_path)

print(f"\nModel saved to: {drive_model_path}")
print("Done!")

# Optional: list saved files
import os
if os.path.exists(drive_model_path):
    print(f"\nSaved files in {drive_model_path}:")
    for file in os.listdir(drive_model_path):
        file_path = os.path.join(drive_model_path, file)
        size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
        print(f"  - {file} ({size:.1f} MB)")

Mounted at /content/drive
SAVING BEST MODEL TO GOOGLE DRIVE


NameError: name 'trainer' is not defined

In [ ]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 13
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for even more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 18
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for more more even more more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 25
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# these are the final additional 5 epochs after this im DONE

# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for more more even more more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 30
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the best model
drive_model_path = "/content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma_69"

print("=" * 60)
print("SAVING BEST MODEL TO GOOGLE DRIVE")
print("=" * 60)
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best F1 macro: {trainer.state.best_metric:.4f}")

# Save model and tokenizer
trainer.save_model(drive_model_path)
tokenizer.save_pretrained(drive_model_path)

print(f"\nModel saved to: {drive_model_path}")
print("Done!")

# Optional: list saved files
import os
if os.path.exists(drive_model_path):
    print(f"\nSaved files in {drive_model_path}:")
    for file in os.listdir(drive_model_path):
        file_path = os.path.join(drive_model_path, file)
        size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
        print(f"  - {file} ({size:.1f} MB)")